# PTQ (FLOAT16)

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## LiteRT 모델로 변환 (PTQ (FLOAT16))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

In [ ]:
tflite_ptq_float16_file = save_dir + 'mnist_ptq_float16.tflite'
open(tflite_ptq_float16_file, 'wb').write(tflite_model)

119448

## File size 비교 : Baseline model vs PQT (FLOAT16) model

In [ ]:
import os
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
print("Size of Baseline LiteRT Model file : {}".format(os.path.getsize(tflite_baseline_model_file)))
print("Size of PTQ(FLOAT16) LiteRT Model file : {}".format(os.path.getsize(tflite_ptq_float16_file)))

Size of Baseline LiteRT Model file : 233596
Size of PTQ(FLOAT16) LiteRT Model file : 119448


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 87.4 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (Baseline model)

In [ ]:
interpreter_base = Interpreter(model_path=str(tflite_baseline_model_file))
interpreter_base.allocate_tensors()

## interpreter 생성 (PTQ (FLOAT16))

In [ ]:
interpreter_ptq_float16 = Interpreter(model_path=str(tflite_ptq_float16_file))
interpreter_ptq_float16.allocate_tensors()

## input/output dtype 확인 (PTQ (FLOAT16))

In [ ]:
input_dtype = interpreter_ptq_float16.get_input_details()[0]['dtype']
output_dtype = interpreter_ptq_float16.get_output_details()[0]['dtype']

print('input dtype: ', input_dtype)
print('output dtype: ', output_dtype)

input dtype:  <class 'numpy.float32'>
output dtype:  <class 'numpy.float32'>


## 추론 실행 (PTQ (FLOAT16))

In [ ]:
test_image = np.expand_dims(test_images[0], axis=0)

input_index = interpreter_ptq_float16.get_input_details()[0]["index"]
output_index = interpreter_ptq_float16.get_output_details()[0]["index"]

interpreter_ptq_float16.set_tensor(input_index, test_image)

interpreter_ptq_float16.invoke()

predictions = interpreter_ptq_float16.get_tensor(output_index)

print(predictions)
print(np.argmax(predictions))
print(test_labels[0])

[[2.7855418e-12 3.3302085e-12 8.9890310e-11 1.5044996e-08 1.1523492e-17
  2.1804226e-14 3.8017534e-19 1.0000000e+00 1.9046969e-12 1.7760205e-11]]
7
7


## Test data 기반 accuracy 평가

In [ ]:
def evaluate_model(interpreter):
  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()

  input_index = input_details[0]["index"]
  output_index = output_details[0]["index"]

  prediction_digits = []
  for test_image in test_images:
    test_image = np.expand_dims(test_image, axis=0)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  accurate_count = 0
  for index in range(len(prediction_digits)):
    if prediction_digits[index] == test_labels[index]:
      accurate_count += 1
  accuracy = accurate_count * 1.0 / len(prediction_digits)

  return accuracy

In [ ]:
print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_ptq_float16))

0.9904
0.9904
